# CS383 — Assignment 1
## Joining and Cleaning Restaurant Inspection Data

*Due: see LMS*

### Scenario
You've been given access to a small relational database of NYC restaurant inspection records, split across three tables:

- `restaurants` — one row per establishment
- `inspections` — one row per inspection visit
- `violations` — one row per violation cited during an inspection

Your task is to write SQL queries that join across these tables, then move into Pandas to clean and summarize what you find.

**This is the first time you're seeing this specific database.** That's intentional — use what you learned about `SELECT`, `WHERE`, `GROUP BY`, joins, and schema-checking (`PRAGMA table_info`, `.dtypes`, `.head()`) to explore it yourself before answering the required questions below. Nothing here has been walked through in class.

---

### Academic integrity and using AI

This assignment is meant to show **your own** understanding of joins and data cleaning — it's the checkpoint that tells you (and your instructor) whether you're ready for Assignment 2 and the capstone, so it needs to reflect what you can actually do without help.

**Allowed** — using an AI tool (Claude, ChatGPT, GitHub Copilot, etc.) as a *tutor*: asking it to explain a concept, explain an error message, explain what a piece of syntax does, or point you toward the right Pandas/SQL method to look up.

**Not allowed** — pasting the assignment questions (or this dataset) into an AI tool and submitting what it gives back; having AI write your queries or code for you; having AI write your reasoning or summary write-ups for you.

A good test: if you can't explain — out loud, without looking anything up — exactly what every line of your submitted notebook does and why, it isn't ready to submit. You may be asked to walk through your code in person; work you can't explain will not receive credit, whether or not AI was involved.

---

### Getting started

Run the cell below once. It builds the three tables above from live NYC Open Data if there's a network connection available, or realistic sample data if not. **Do not skip or modify this cell** — everything after it assumes these three tables exist in `restaurant_inspections.db`.

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    raw = pd.read_csv(raw_path).head(15000)
    if "score" in raw.columns:
        raw["score"] = pd.to_numeric(raw["score"], errors="coerce")

    restaurants = raw[["camis", "dba", "boro", "cuisine_description"]].drop_duplicates(
        subset="camis"
    ).reset_index(drop=True)

    inspections = raw[["camis", "inspection_date", "action", "score", "grade"]].drop_duplicates(
        subset=["camis", "inspection_date"]
    ).reset_index(drop=True)
    inspections.insert(0, "inspection_id", range(1, len(inspections) + 1))

    with_ids = raw.merge(
        inspections[["camis", "inspection_date", "inspection_id"]], on=["camis", "inspection_date"]
    )
    violations = with_ids.loc[
        with_ids["violation_code"].notna(),
        ["inspection_id", "violation_code", "violation_description", "critical_flag"],
    ].reset_index(drop=True)

    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)

    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza",
                       "Japanese", "Caribbean", "Bakery", "Coffee/Tea", "Chicken"]

    n_restaurants = 150
    camis_ids = np.arange(50_000_001, 50_000_001 + n_restaurants)
    restaurants = pd.DataFrame({
        "camis": camis_ids,
        "dba": [f"Restaurant {i + 1}" for i in range(n_restaurants)],
        "boro": rng.choice(boroughs_list, size=n_restaurants, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n_restaurants),
    })

    # Deliberately messy cuisine text on a chunk of rows (inconsistent case/whitespace).
    messy_idx = rng.choice(restaurants.index, size=20, replace=False)
    def _messify(s):
        pick = rng.integers(0, 3)
        if pick == 0:
            return s.upper()
        elif pick == 1:
            return f"  {s.lower()}  "
        return s.lower()
    restaurants.loc[messy_idx, "cuisine_description"] = restaurants.loc[
        messy_idx, "cuisine_description"
    ].apply(_messify)

    # A handful of exact duplicate restaurant rows.
    restaurants = pd.concat(
        [restaurants, restaurants.sample(5, random_state=1)], ignore_index=True
    )

    n_inspections = 400
    inspections = pd.DataFrame({
        "inspection_id": np.arange(1, n_inspections + 1),
        "camis": rng.choice(camis_ids, size=n_inspections),
        "inspection_date": pd.date_range("2025-01-01", periods=n_inspections, freq="18h").astype(str),
        "action": "Violations were cited in the following area(s).",
        "score": rng.integers(0, 60, size=n_inspections),
        "grade": rng.choice(["A", "B", "C", None], size=n_inspections, p=[0.55, 0.25, 0.10, 0.10]),
    })
    # A few duplicate inspection rows too.
    inspections = pd.concat(
        [inspections, inspections.sample(4, random_state=2)], ignore_index=True
    )

    violation_pool = [
        ("04L", "Evidence of mice or live mice present", "Critical"),
        ("06C", "Food not protected from potential source of contamination", "Critical"),
        ("08A", "Facility not vermin proof", "Not Critical"),
        ("10F", "Non-food contact surface improperly constructed", "Not Critical"),
        ("02B", "Hot food item not held at or above 140 F", "Critical"),
        ("04N", "Filth flies or food/refuse/sewage-associated flies present", "Critical"),
        ("06D", "Food contact surface not properly washed", "Critical"),
        ("05D", "Hand washing facility not accessible", "Critical"),
    ]
    n_violations = 700
    picks = rng.integers(0, len(violation_pool), size=n_violations)
    violations = pd.DataFrame({
        "inspection_id": rng.choice(inspections["inspection_id"].unique(), size=n_violations),
        "violation_code": [violation_pool[i][0] for i in picks],
        "violation_description": [violation_pool[i][1] for i in picks],
        "critical_flag": [violation_pool[i][2] for i in picks],
    })

    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data loaded.")
print(f"restaurants: {len(restaurants):,} rows | inspections: {len(inspections):,} rows | violations: {len(violations):,} rows")

conn = sqlite3.connect("restaurant_inspections.db")
restaurants.to_sql("restaurants", conn, if_exists="replace", index=False)
inspections.to_sql("inspections", conn, if_exists="replace", index=False)
violations.to_sql("violations", conn, if_exists="replace", index=False)

print("Tables ready in restaurant_inspections.db: restaurants, inspections, violations")

Shared snapshot data loaded.
restaurants: 4,279 rows | inspections: 4,529 rows | violations: 14,867 rows
Tables ready in restaurant_inspections.db: restaurants, inspections, violations


---

### Explore before you answer

Before jumping into the required questions, take a few minutes to look around:

- What columns does each table have, and what do they look like? (`PRAGMA table_info(...)`, `.dtypes`, `.head()`)
- How do the three tables relate to each other — which columns would you join on?
- Do you notice anything odd or messy in the data as you look at it?

Use the scratch cell below (and add more cells if you want) to explore. This part isn't graded directly, but it'll make the required questions much easier.

In [2]:
# Scratch space -- explore restaurants, inspections, and violations here.
print(type(raw))
raw.head(5)


<class 'pandas.DataFrame'>


,camis,dba,boro,building,street,zipcode,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,record_date,inspection_type,latitude,longitude,score,grade,grade_date
0,50141699,FRESH PIZZA & DELI,Manhattan,876,6 AVENUE,10001.0,Pizza,2026-07-30T00:00:00.000,Violations were cited in the following area(s).,10F,Non-food contact surface or equipment made of ...,Not Critical,2026-09-04T06:00:16.000,Cycle Inspection / Initial Inspection,40.747976,-73.989090,11.0,A,2026-07-30T00:00:00.000
1,50102634,MR. ZENG KITCHEN,Queens,114-02,LIBERTY AVENUE,11419.0,Chinese,2026-01-28T00:00:00.000,Violations were cited in the following area(s).,04K,Evidence of rats or live rats in establishment...,Critical,2026-09-04T06:00:16.000,Cycle Inspection / Initial Inspection,40.685651,-73.828478,12.0,A,2026-01-28T00:00:00.000
2,50150011,SERENA BISTRO,Staten Island,96,PAGE AVENUE,10309.0,New American,2026-02-21T00:00:00.000,Violations were cited in the following area(s).,06B,"Tobacco or electronic cigarette use, eating, o...",Critical,2026-09-04T06:00:16.000,Pre-permit (Operational) / Initial Inspection,40.521891,-74.235023,23.0,N,NaN
3,50171396,KNORI HANDROLL BAR,Brooklyn,121,GREENPOINT AVENUE,11222.0,Japanese,2026-01-31T00:00:00.000,Violations were cited in the following area(s).,06A,Personal cleanliness is inadequate. Outer garm...,Critical,2026-09-04T06:00:16.000,Pre-permit (Operational) / Initial Inspection,40.730094,-73.955371,25.0,N,NaN
4,50177723,MR. CEBA'S,Queens,74-20,ROOSEVELT AVENUE,11372.0,Spanish,2025-12-29T00:00:00.000,Violations were cited in the following area(s).,06C,"Food, supplies, or equipment not protected fro...",Critical,2026-09-04T06:00:16.000,Pre-permit (Operational) / Initial Inspection,40.746860,-73.891156,13.0,A,2025-12-29T00:00:00.000


In [3]:
#Restaurants Table
pd.read_sql_query("SELECT * FROM restaurants LIMIT 5;", conn)


,camis,dba,boro,cuisine_description
0,50141699,FRESH PIZZA & DELI,Manhattan,Pizza
1,50102634,MR. ZENG KITCHEN,Queens,Chinese
2,50150011,SERENA BISTRO,Staten Island,New American
3,50171396,KNORI HANDROLL BAR,Brooklyn,Japanese
4,50177723,MR. CEBA'S,Queens,Spanish


In [4]:
#Inspections Table
pd.read_sql_query("SELECT * FROM inspections LIMIT 5;", conn)


,inspection_id,camis,inspection_date,action,score,grade
0,1,50141699,2026-07-30T00:00:00.000,Violations were cited in the following area(s).,11.0,A
1,2,50102634,2026-01-28T00:00:00.000,Violations were cited in the following area(s).,12.0,A
2,3,50150011,2026-02-21T00:00:00.000,Violations were cited in the following area(s).,23.0,N
3,4,50171396,2026-01-31T00:00:00.000,Violations were cited in the following area(s).,25.0,N
4,5,50177723,2025-12-29T00:00:00.000,Violations were cited in the following area(s).,13.0,A


In [5]:
#Violations Table
pd.read_sql_query("SELECT * FROM violations LIMIT 5;", conn)


,inspection_id,violation_code,violation_description,critical_flag
0,1,10F,Non-food contact surface or equipment made of ...,Not Critical
1,2,04K,Evidence of rats or live rats in establishment...,Critical
2,3,06B,"Tobacco or electronic cigarette use, eating, o...",Critical
3,4,06A,Personal cleanliness is inadequate. Outer garm...,Critical
4,5,06C,"Food, supplies, or equipment not protected fro...",Critical


In [6]:
#Data Types

print(raw.dtypes)

camis                      int64
dba                          str
boro                         str
building                     str
street                       str
zipcode                  float64
cuisine_description          str
inspection_date              str
action                       str
violation_code               str
violation_description        str
critical_flag                str
record_date                  str
inspection_type              str
latitude                 float64
longitude                float64
score                    float64
grade                        str
grade_date                   str
dtype: object


---

## Required Deliverable 1 — Restaurants Per Cuisine, Cleaned

The `cuisine_description` column has inconsistent capitalization and stray whitespace in places, and the `restaurants` table has a few exact duplicate rows.

Clean both issues in Pandas, then produce a count of restaurants per (cleaned) cuisine type, sorted from most to fewest.

In [7]:
# Your code here.

restaurants[["cuisine_description"]].head(20)

,cuisine_description
0,Pizza
1,Chinese
2,New American
3,Japanese
4,Spanish
5,Bakery Products/Desserts
6,American
7,Thai
8,African
9,Latin American


In [8]:
# Check for Missing Values
restaurants.isna().sum()


camis                  0
dba                    0
boro                   0
cuisine_description    0
dtype: int64

In [9]:
# Checking for Inconsistent Text
# I don't see white space or capitalization problem 

print(restaurants["cuisine_description"].unique())

<StringArray>
[                         'Pizza',                        'Chinese',
                   'New American',                       'Japanese',
                        'Spanish',       'Bakery Products/Desserts',
                       'American',                           'Thai',
                        'African',                 'Latin American',
                        'Italian',                          'Irish',
                  'Mediterranean',                     'Coffee/Tea',
                          'Greek',                        'Mexican',
                      'Caribbean',                         'Indian',
                'Southeast Asian',                    'Bangladeshi',
 'Juice, Smoothies, Fruit Salads',                'Frozen Desserts',
                        'Chicken',                       'Barbecue',
                     'Sandwiches',                          'Other',
                      'Pakistani',                  'Jewish/Kosher',
             'Asian/

In [10]:
# Capitalized and Whitespace Removed

restaurants["cuisine_description"] = restaurants["cuisine_description"].str.strip().str.upper()
print(restaurants["cuisine_description"].unique())


<StringArray>
[                         'PIZZA',                        'CHINESE',
                   'NEW AMERICAN',                       'JAPANESE',
                        'SPANISH',       'BAKERY PRODUCTS/DESSERTS',
                       'AMERICAN',                           'THAI',
                        'AFRICAN',                 'LATIN AMERICAN',
                        'ITALIAN',                          'IRISH',
                  'MEDITERRANEAN',                     'COFFEE/TEA',
                          'GREEK',                        'MEXICAN',
                      'CARIBBEAN',                         'INDIAN',
                'SOUTHEAST ASIAN',                    'BANGLADESHI',
 'JUICE, SMOOTHIES, FRUIT SALADS',                'FROZEN DESSERTS',
                        'CHICKEN',                       'BARBECUE',
                     'SANDWICHES',                          'OTHER',
                      'PAKISTANI',                  'JEWISH/KOSHER',
             'ASIAN/

In [11]:
# Checking For Restaurants Duplicates
# No Duplicates for some reason

print(f"Duplicate Rows: {restaurants.duplicated().sum()}")

Duplicate Rows: 0


In [12]:
# Checking For Restaurants Duplicates

restaurants[restaurants.duplicated(keep=False)].sort_values("camis")

,camis,dba,boro,cuisine_description


In [13]:
#Remove Restaurants Duplicates

restaurants = restaurants.drop_duplicates()
restaurants.duplicated().sum()

np.int64(0)

In [14]:
# Check Restaurants Duplicates Again

restaurants = restaurants.drop_duplicates().reset_index(drop=True)
print(f"Rows after dropping duplicates: {len(restaurants):,}")

Rows after dropping duplicates: 4,279


In [15]:
# Count for Restaurants per Cuisine Type from Highest to Lowest

restaurants.groupby("cuisine_description").size().sort_values(ascending=False)

cuisine_description
AMERICAN              643
CHINESE               399
COFFEE/TEA            379
PIZZA                 238
LATIN AMERICAN        196
                     ... 
ENGLISH                 1
LEBANESE                1
ETHIOPIAN               1
NUTS/CONFECTIONARY      1
SCANDINAVIAN            1
Length: 82, dtype: int64

---

## Required Deliverable 2 — Worst Inspection Scores, With Restaurant Names (JOIN required)

Using a SQL `JOIN` across `restaurants` and `inspections`, find the 10 individual inspections with the **highest** score (a higher score means more violations) among inspections that also received a letter grade.

Your result should include: restaurant name, borough, cuisine, inspection date, score, and grade.

In [16]:
# Your code here.

sql = """
SELECT restaurants.dba, restaurants.boro, restaurants.cuisine_description, inspections.inspection_date, inspections.score, inspections.grade
FROM restaurants
JOIN inspections ON restaurants.camis = inspections.camis
ORDER BY inspections.score DESC
LIMIT 10;
"""
pd.read_sql_query(sql, conn)

,dba,boro,cuisine_description,inspection_date,score,grade
0,% ARABICA,Manhattan,Middle Eastern,2026-03-31T00:00:00.000,163.0,NaN
1,KING SUNSHINE #2 JERK CENTER,Bronx,Caribbean,2026-04-30T00:00:00.000,160.0,NaN
2,STEINBERG'S KOSHER BAKERY,Brooklyn,Jewish/Kosher,2025-12-23T00:00:00.000,143.0,N
3,PANAJACHEL SOLOLA,Staten Island,Latin American,2025-11-24T00:00:00.000,138.0,NaN
4,CARIBBEAN CRAVE RESTAURANT,Queens,Caribbean,2026-01-28T00:00:00.000,131.0,N
5,PANAJACHEL SOLOLA,Brooklyn,Other,2026-08-26T00:00:00.000,130.0,N
6,CANTINA HARLEM,Manhattan,Tex-Mex,2026-05-28T00:00:00.000,125.0,NaN
7,AL REEF BAKERY & RESTAURANT,Brooklyn,Middle Eastern,2026-04-30T00:00:00.000,124.0,NaN
8,NEOVISTA FRESH MARKET,Brooklyn,American,2026-03-30T00:00:00.000,124.0,NaN
9,SU & GAO,Brooklyn,Chinese,2026-02-26T00:00:00.000,120.0,NaN


---

## Required Deliverable 3 — Most Common Critical Violations, By Cuisine (JOIN required, all 3 tables)

Pick one cuisine type. Using SQL `JOIN`s across all three tables, find the 5 most common `"Critical"` violation descriptions among restaurants of that cuisine.

Then reproduce the same result using Pandas `.merge()` instead of SQL, and confirm the two approaches agree.

In [17]:
# SQL version -- your code here.

sql = """
SELECT restaurants.cuisine_description, violations.critical_flag, violations.violation_description, COUNT(*) as critical_count
FROM restaurants
JOIN inspections ON restaurants.camis = inspections.camis
JOIN violations ON inspections.inspection_id = violations.inspection_id
WHERE violations.critical_flag = "Critical" AND restaurants.cuisine_description = 'American'
GROUP BY violation_description
ORDER BY critical_count DESC
LIMIT 5;
"""
sql_results = pd.read_sql_query(sql, conn)
sql_results

,cuisine_description,critical_flag,violation_description,critical_count
0,American,Critical,"Food contact surface not properly washed, rins...",178
1,American,Critical,"Food, supplies, or equipment not protected fro...",145
2,American,Critical,Cold TCS food item held above 41 °F; smoked or...,138
3,American,Critical,Filth flies or food/refuse/sewage associated w...,91
4,American,Critical,"Raw, cooked or prepared food is adulterated, c...",83


In [18]:
# Pandas version -- your code here.
merged = (
    restaurants
    .merge(inspections, on="camis")
    .merge(violations, on="inspection_id")
)

critical_count = (
    merged[(merged["cuisine_description"] == "AMERICAN") & (merged["critical_flag"] == "Critical")]
    .groupby([
        "cuisine_description",
        "critical_flag",
        "violation_description"
    ])
    .size()
    .reset_index(name="critical_count")
    .sort_values("critical_count", ascending=False)
)

critical_count.head(5)


,cuisine_description,critical_flag,violation_description,critical_count
7,AMERICAN,Critical,"Food contact surface not properly washed, rins...",178
14,AMERICAN,Critical,"Food, supplies, or equipment not protected fro...",145
1,AMERICAN,Critical,Cold TCS food item held above 41 °F; smoked or...,138
5,AMERICAN,Critical,Filth flies or food/refuse/sewage associated w...,91
27,AMERICAN,Critical,"Raw, cooked or prepared food is adulterated, c...",83


In [19]:
# Confirm the SQL and Pandas results agree -- your code here.
# pandas_results = critical_counts.sort_values(
#     ["violation_description"]
# ).reset_index(drop=True)

# sql_results = sql_results.sort_values(
#     ["violation_description"]
# ).reset_index(drop=True)


pandas_compare = critical_count.head(5)[
    ["violation_description", "critical_count"]
]

sql_compare = sql_results[
    ["violation_description", "critical_count"]
]

comparison = pandas_compare.merge(
    sql_compare,
    on=["violation_description", "critical_count"],
    how="outer",
    indicator=True
).sort_values("critical_count", ascending=False)

comparison



,violation_description,critical_count,_merge
2,"Food contact surface not properly washed, rins...",178,both
3,"Food, supplies, or equipment not protected fro...",145,both
0,Cold TCS food item held above 41 °F; smoked or...,138,both
1,Filth flies or food/refuse/sewage associated w...,91,both
4,"Raw, cooked or prepared food is adulterated, c...",83,both


---

## Required Deliverable 4 — Handle the Missing Grades

Not every inspection has a letter grade — some are still missing (`None` / `NaN`).

Decide how to handle this for the purpose of computing an **average score by grade**: drop the missing rows? Label them `"Ungraded"` and keep them separate? Something else? Write one or two sentences explaining your reasoning *before* your code, then implement it.

**Your reasoning:**

I would label them "Ungraded" and keep them separate because though some have NaN as a grade they still have scores which could be a useful information.

In [20]:
# Your code here.

#Check for # of NaN

inspections["grade"].isna().value_counts()


grade
False    3022
True     1507
Name: count, dtype: int64

In [21]:
# Turns all NaN Grades into Ungraded
inspections["grade"] = inspections["grade"].fillna("Ungraded")

inspections["grade"].value_counts()

grade
A           2263
Ungraded    1507
N            310
Z            212
B            149
C             73
P             15
Name: count, dtype: int64

---

## Required Deliverable 5 — Summary for a Non-Technical Reader

Pick one finding from your analysis above that an actual restaurant-goer (not a data scientist) would care about.

Produce one clean summary table or chart, and write a two-to-three sentence explanation in plain language — no jargon, no code terms. Imagine you're telling a friend, not writing a lab report.

In [34]:
# Your code here.

pd.crosstab(restaurants["cuisine_description"], violations["critical_flag"]).sort_values("Critical", ascending=False)

critical_flag,Critical,Not Applicable,Not Critical
cuisine_description,,,
AMERICAN,336,6,301
CHINESE,205,5,189
COFFEE/TEA,202,1,176
PIZZA,141,1,96
LATIN AMERICAN,104,2,90
...,...,...,...
AFGHAN,0,0,1
CHINESE/CUBAN,0,0,1
CAJUN,0,0,1


**Your summary (plain language):**
The cuisine type with the most number of violations that are critical is American.


---

## Grading Rubric

| Criterion | Points |
|---|---|
| Deliverable 1 — cleaning is correct and counts are right | 15 |
| Deliverable 2 — JOIN is correct and produces the right 10 rows | 20 |
| Deliverable 3 — 3-table JOIN correct; SQL and Pandas results genuinely match | 25 |
| Deliverable 4 — reasoning is stated and implementation matches it | 15 |
| Deliverable 5 — finding is genuinely non-technical, table/chart is clear | 15 |
| Notebook runs top to bottom without errors | 10 |
| **Total** | **100** |

---

## Submission

### One-time GitHub setup (reused for every assignment this semester)

1. On github.com, create one new **public** repository named `cs383-assignments`. You'll keep reusing this same repo all semester — one folder per assignment, not a new repo each time.
2. Inside it, create a folder for this assignment: `assignment1/`.
3. Put your completed notebook in that folder, keeping its original filename: `assignment1_restaurant_inspections.ipynb`.

If you've never pushed to this repo before, from inside your project folder run:

```bash
git init
git remote add origin <your-repo-URL>
git branch -M main
```

(Prefer a GUI? GitHub Desktop or VS Code's built-in Git panel do the same thing without the command line.)

### Every time you submit

```bash
git add -A
git commit -m "Assignment 1: restaurant inspections"
git push
```

Submit the **direct link to your notebook file on GitHub** (not just the repo homepage — the URL should end in `assignment1/assignment1_restaurant_inspections.ipynb`) on BrightSpace under **Assignment 1**.

Make sure your notebook runs top to bottom without errors before you submit — a notebook that only works because cells were run out of order will not be graded as passing.